In [1]:
!pip install -q streamlit lightgbm xgboost altair pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 34.1 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()

Saving retail_store_inventory.csv to retail_store_inventory.csv


In [3]:
%%writefile model.py
"""
ML model: train LightGBM on retail inventory data and forecast future week demand.
"""
import os
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error

DATA_PATH = "retail_store_inventory.csv"

W_DSG, W_STS, W_PIS = 0.45, 0.30, 0.25


def load_data() -> pd.DataFrame:
    df = pd.read_csv(DATA_PATH)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values(["Store ID", "Product ID", "Date"]).reset_index(drop=True)
    return df


def _build_weekly(df: pd.DataFrame):
    """Aggregate daily data to weekly store-category level and add lag features."""
    promo_cols = [c for c in df.columns if "promo" in c.lower() or "holiday" in c.lower()]
    PROMO_COL = promo_cols[0] if promo_cols else None

    df = df.copy()
    df["Week"] = df["Date"].dt.to_period("W").dt.start_time

    weekly = df.groupby(["Store ID", "Category", "Week"]).agg(
        Units_Sold=("Units Sold", "sum"),
        Inventory_Level=("Inventory Level", "mean"),
        Price=("Price", "mean"),
        Discount=("Discount", "mean"),
    ).reset_index()

    if PROMO_COL:
        pw = df.groupby(["Store ID", "Category", "Week"])[PROMO_COL].mean().reset_index()
        weekly = weekly.merge(pw, on=["Store ID", "Category", "Week"])

    weekly = weekly.sort_values(["Store ID", "Category", "Week"])
    g = weekly.groupby(["Store ID", "Category"])["Units_Sold"]
    weekly["Lag_1"] = g.shift(1)
    weekly["Rolling_4"] = g.transform(lambda x: x.shift(1).rolling(4, min_periods=1).mean())
    weekly["Week_of_Year"] = weekly["Week"].dt.isocalendar().week.astype(int)
    weekly["Month"] = weekly["Week"].dt.month
    weekly = weekly.dropna(subset=["Lag_1", "Rolling_4"]).reset_index(drop=True)
    return weekly, PROMO_COL


def train_model(df: pd.DataFrame) -> dict:
    """Train LightGBM, return model bundle with all metadata needed for forecasting."""
    weekly, PROMO_COL = _build_weekly(df)

    le_store = LabelEncoder()
    le_cat = LabelEncoder()
    weekly["Store_enc"] = le_store.fit_transform(weekly["Store ID"])
    weekly["Cat_enc"] = le_cat.fit_transform(weekly["Category"])

    feats = ["Store_enc", "Cat_enc", "Inventory_Level", "Price", "Discount",
             "Lag_1", "Rolling_4", "Week_of_Year", "Month"]
    if PROMO_COL:
        feats.append(PROMO_COL)

    split_date = weekly["Week"].quantile(0.80)
    train_df = weekly[weekly["Week"] < split_date]
    test_df = weekly[weekly["Week"] >= split_date]

    model = lgb.LGBMRegressor(n_estimators=250, max_depth=5, learning_rate=0.05, verbosity=-1)
    model.fit(train_df[feats], train_df["Units_Sold"])

    pred = model.predict(test_df[feats])
    y_true = test_df["Units_Sold"].values
    mae = mean_absolute_error(y_true, pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, pred)))
    wape = float(np.sum(np.abs(y_true - pred)) / np.sum(y_true) * 100)

    # Promotion uplift per category
    promo_uplift: dict = {}
    if PROMO_COL:
        promo_flag = df[PROMO_COL]
        promo_bool = (
            promo_flag.astype(str).str.lower().isin(["yes", "1", "true"])
            if promo_flag.dtype == object else promo_flag.astype(bool)
        )
        df2 = df.copy()
        df2["_pb"] = promo_bool
        pg = df2.groupby(["Category", "_pb"])["Units Sold"].mean().unstack()
        if pg.shape[1] == 2:
            pg.columns = ["No_Promo", "Promo"]
            pg["Uplift_%"] = (pg["Promo"] - pg["No_Promo"]) / pg["No_Promo"] * 100
            promo_uplift = pg["Uplift_%"].to_dict()

    # Per-product sales trend (last 14 days vs prior 14 days)
    def compute_trend(grp):
        grp = grp.sort_values("Date")
        recent = grp["Units Sold"].tail(14).mean()
        prior = grp["Units Sold"].tail(28).head(14).mean()
        return 0.0 if (not prior or pd.isna(prior)) else float((recent - prior) / prior)

    trend_map = (
        df.groupby(["Store ID", "Product ID"], group_keys=False)
          .apply(compute_trend)
    )
    trend_map.name = "Trend_Pct"

    # Product sales share within category (for distributing category forecast)
    product_totals = (
        df.groupby(["Store ID", "Category", "Product ID"])["Units Sold"].sum().reset_index()
    )
    cat_totals = product_totals.groupby(["Store ID", "Category"])["Units Sold"].transform("sum")
    product_totals["Share"] = product_totals["Units Sold"] / cat_totals.replace(0, np.nan)

    # Latest snapshot per product (inventory, category)
    product_latest = (
        df.sort_values("Date")
          .groupby(["Store ID", "Product ID"])
          .tail(1)[["Store ID", "Product ID", "Category", "Inventory Level"]]
          .copy()
    )

    # Feature importance (gain-based, mirrors SHAP bar chart)
    importance_df = pd.DataFrame({
        "Feature": feats,
        "Importance": model.feature_importances_,
    }).sort_values("Importance", ascending=False).reset_index(drop=True)

    # Friendly display names for features
    feature_labels = {
        "Store_enc": "Store",
        "Cat_enc": "Category",
        "Inventory_Level": "Inventory Level",
        "Price": "Price",
        "Discount": "Discount",
        "Lag_1": "Lag 1 Week",
        "Rolling_4": "Rolling 4-Week Avg",
        "Week_of_Year": "Week of Year",
        "Month": "Month",
    }
    if PROMO_COL:
        feature_labels[PROMO_COL] = "Promotion / Holiday"
    importance_df["Feature Label"] = importance_df["Feature"].map(
        lambda f: feature_labels.get(f, f)
    )

    # Test actuals vs predictions (for Forecast vs Actual chart)
    test_plot = test_df[["Store ID", "Category", "Week", "Units_Sold"]].copy()
    test_plot["Predicted"] = pred.clip(0)
    test_plot = test_plot.rename(columns={"Units_Sold": "Actual"})

    return {
        "model": model,
        "le_store": le_store,
        "le_cat": le_cat,
        "feats": feats,
        "PROMO_COL": PROMO_COL,
        "weekly": weekly,
        "metrics": {"MAE": mae, "RMSE": rmse, "WAPE": wape},
        "promo_uplift": promo_uplift,
        "trend_map": trend_map,
        "product_totals": product_totals,
        "product_latest": product_latest,
        "importance_df": importance_df,
        "test_plot": test_plot,
    }


def _dsg_score(ratio: float) -> float:
    if pd.isna(ratio):
        return 50.0
    if ratio < 0.8:
        return 90.0
    if ratio <= 1.5:
        return float(np.interp(ratio, [0.8, 1.5], [90, 50]))
    if ratio <= 2.5:
        return float(np.interp(ratio, [1.5, 2.5], [50, 10]))
    return 10.0


def _classify(score: float) -> str:
    if score >= 70:
        return "Grow"
    if score >= 40:
        return "Monitor"
    return "Reduce"


def _primary_driver(row) -> str:
    contribs = {
        "Inventory Gap": row["DSG_contribution"],
        "Sales Trend": row["STS_contribution"],
        "Promotion Impact": row["PIS_contribution"],
    }
    driver = max(contribs, key=lambda k: abs(contribs[k]))
    direction = "pushing up" if contribs[driver] > 0 else "pushing down"
    return f"{driver} ({direction})"


def forecast_weeks(
    df: pd.DataFrame,
    model_data: dict,
    n_weeks: int = 1,
    store_filter: str | None = None,
    category_filter: str | None = None,
    overrides: dict | None = None,
) -> pd.DataFrame:
    """
    Forecast demand for the next n_weeks weeks using chained predictions.
    Returns a single DataFrame with a 'Week' column (1, 2, 3...).
    """
    model = model_data["model"]
    le_store = model_data["le_store"]
    le_cat = model_data["le_cat"]
    feats = model_data["feats"]
    PROMO_COL = model_data["PROMO_COL"]
    promo_uplift = model_data["promo_uplift"]
    trend_map = model_data["trend_map"]
    product_totals = model_data["product_totals"]
    product_latest = model_data["product_latest"]

    # Build product snapshot with trend
    snapshot = product_latest.merge(
        product_totals[["Store ID", "Category", "Product ID", "Share"]],
        on=["Store ID", "Category", "Product ID"], how="left"
    )
    trend_reset = trend_map.reset_index()
    trend_reset.columns = ["Store ID", "Product ID", "Trend_Pct"]
    snapshot = snapshot.merge(trend_reset, on=["Store ID", "Product ID"], how="left")
    snapshot["Trend_Pct"] = snapshot["Trend_Pct"].fillna(0.0)

    # Apply filters
    if store_filter:
        snapshot = snapshot[snapshot["Store ID"] == store_filter].copy()
    if category_filter:
        snapshot = snapshot[snapshot["Category"] == category_filter].copy()

    if snapshot.empty:
        return pd.DataFrame()

    # Working copy of weekly data for chained forecasting
    current_weekly = model_data["weekly"].copy()

    all_rows: list[pd.DataFrame] = []
    accumulated_demand: dict = {}  # (Store ID, Product ID) -> total demand so far

    for week_num in range(1, n_weeks + 1):
        # Get latest weekly row per store-category for input features
        latest_sc = (
            current_weekly.sort_values("Week")
                          .groupby(["Store ID", "Category"])
                          .tail(1)
                          .copy()
        )

        # Apply store/category filter to latest_sc too
        if store_filter:
            latest_sc = latest_sc[latest_sc["Store ID"] == store_filter]
        if category_filter:
            latest_sc = latest_sc[latest_sc["Category"] == category_filter]

        # Apply user overrides to features
        if overrides:
            for col, val in overrides.items():
                if col in latest_sc.columns:
                    latest_sc[col] = val

        # Advance to next week
        max_week = latest_sc["Week"].max()
        next_week_ts = max_week + pd.Timedelta(weeks=1)
        latest_sc = latest_sc.copy()
        latest_sc["Week"] = next_week_ts
        latest_sc["Week_of_Year"] = int(next_week_ts.isocalendar()[1])
        latest_sc["Month"] = next_week_ts.month

        # Encode store / category (handle unseen labels gracefully)
        store_classes = list(le_store.classes_)
        cat_classes = list(le_cat.classes_)
        latest_sc["Store_enc"] = latest_sc["Store ID"].apply(
            lambda s: le_store.transform([s])[0] if s in store_classes else 0
        )
        latest_sc["Cat_enc"] = latest_sc["Category"].apply(
            lambda c: le_cat.transform([c])[0] if c in cat_classes else 0
        )

        latest_sc["Cat_Forecast"] = model.predict(latest_sc[feats]).clip(0)

        # Merge category forecast into product snapshot
        week_snap = snapshot.merge(
            latest_sc[["Store ID", "Category", "Cat_Forecast"]],
            on=["Store ID", "Category"], how="left"
        ).copy()

        week_snap["Forecasted_Demand"] = (week_snap["Share"] * week_snap["Cat_Forecast"]).clip(0)
        week_snap["Week"] = week_num

        # Adjust inventory for coverage ratio (deplete by prior weeks' demand)
        week_snap["Remaining_Inventory"] = week_snap.apply(
            lambda r: max(0, r["Inventory Level"] - accumulated_demand.get(
                (r["Store ID"], r["Product ID"]), 0.0
            )),
            axis=1
        )

        # Override inventory if user specified
        if overrides and "Inventory_Level" in overrides:
            week_snap["Remaining_Inventory"] = overrides["Inventory_Level"]

        week_snap["Coverage_Ratio"] = (
            week_snap["Remaining_Inventory"] / week_snap["Forecasted_Demand"].replace(0, np.nan)
        )

        # Decision Intelligence Scores
        week_snap["DSG"] = week_snap["Coverage_Ratio"].apply(_dsg_score)
        week_snap["STS"] = (50 + week_snap["Trend_Pct"] * 100).clip(0, 100)

        week_snap["PIS"] = week_snap["Category"].map(
            lambda c: min(100, max(0, 50 + promo_uplift.get(c, 0)))
        )
        if overrides and "Holiday/Promotion" in overrides:
            week_snap["PIS"] = 60.0 if overrides["Holiday/Promotion"] == 1 else 50.0

        week_snap["Decision_Score"] = (
            W_DSG * week_snap["DSG"] + W_STS * week_snap["STS"] + W_PIS * week_snap["PIS"]
        )
        week_snap["Decision_Tier"] = week_snap["Decision_Score"].apply(_classify)

        week_snap["DSG_contribution"] = W_DSG * (week_snap["DSG"] - 50)
        week_snap["STS_contribution"] = W_STS * (week_snap["STS"] - 50)
        week_snap["PIS_contribution"] = W_PIS * (week_snap["PIS"] - 50)
        week_snap["Primary_Driver"] = week_snap.apply(_primary_driver, axis=1)

        all_rows.append(week_snap)

        # Accumulate demand for next week's inventory depletion
        for _, row in week_snap.iterrows():
            key = (row["Store ID"], row["Product ID"])
            accumulated_demand[key] = accumulated_demand.get(key, 0.0) + row["Forecasted_Demand"]

        # Update current_weekly with this week's category predictions for chaining
        new_rows = latest_sc.copy()
        new_rows["Units_Sold"] = latest_sc["Cat_Forecast"]
        current_weekly = pd.concat([current_weekly, new_rows], ignore_index=True)
        current_weekly = current_weekly.sort_values(["Store ID", "Category", "Week"])
        g = current_weekly.groupby(["Store ID", "Category"])["Units_Sold"]
        current_weekly["Lag_1"] = g.shift(1)
        current_weekly["Rolling_4"] = g.transform(
            lambda x: x.shift(1).rolling(4, min_periods=1).mean()
        )
        current_weekly = current_weekly.dropna(subset=["Lag_1", "Rolling_4"]).reset_index(drop=True)

    return pd.concat(all_rows, ignore_index=True)


Writing model.py


In [4]:
%%writefile app.py
"""
Sales Demand Forecasting Dashboard
AI-powered weekly demand prediction with Decision Intelligence scoring + Gemini LLM insights.
"""
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import sys, os

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from model import load_data, train_model, forecast_weeks
from llm_summary import generate_summary, answer_question

# ─────────────────────────────────────────────
# Page config
# ─────────────────────────────────────────────
st.set_page_config(
    page_title="Sales Demand Forecasting",
    page_icon="📈",
    layout="wide",
    initial_sidebar_state="expanded",
)

TIER_COLORS = {"Grow": "#22c55e", "Monitor": "#f59e0b", "Reduce": "#ef4444"}
TIER_BG    = {"Grow": "#d1fae5", "Monitor": "#fef3c7", "Reduce": "#fee2e2"}
TIER_FG    = {"Grow": "#065f46", "Monitor": "#92400e", "Reduce": "#991b1b"}

STORES     = ["All", "S001", "S002", "S003", "S004", "S005"]
CATEGORIES = ["All", "Clothing", "Electronics", "Furniture", "Groceries", "Toys"]


# ─────────────────────────────────────────────
# Cached model training
# ─────────────────────────────────────────────
@st.cache_resource(show_spinner="⏳ Training LightGBM on 73 K rows — one moment…")
def get_model_bundle():
    df = load_data()
    model_data = train_model(df)
    return df, model_data


# ─────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────
def tier_badge(tier: str) -> str:
    icon = {"Grow": "🟢", "Monitor": "🟡", "Reduce": "🔴"}.get(tier, "⚪")
    return f"{icon} {tier}"


def render_metric_card(label, value, delta=None, delta_color="normal"):
    st.metric(label=label, value=value, delta=delta, delta_color=delta_color)


# ─────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────
def main():
    # ── Header ──
    st.title("📈 Sales Demand Forecasting Dashboard")
    st.caption(
        "Trained on 73,100 retail records · LightGBM · Decision Intelligence (Grow / Monitor / Reduce) · "
        "Gemini AI summaries"
    )
    st.divider()

    # ── Load model ──
    df, model_data = get_model_bundle()
    metrics = model_data["metrics"]

    # ─────────────────────────────────────────
    # Sidebar
    # ─────────────────────────────────────────
    with st.sidebar:
        st.header("⚙️ Forecast Settings")

        n_weeks = st.radio(
            "Forecast Horizon",
            options=[1, 2, 3],
            format_func=lambda x: f"Week +{x}",
            horizontal=True,
            help="How many weeks ahead to predict",
        )

        st.subheader("🔍 Filters")
        store_filter    = st.selectbox("Store", STORES)
        category_filter = st.selectbox("Category", CATEGORIES)

        st.subheader("🎛️ Feature Overrides")
        st.caption("Set to 0 / 'Use actual' to keep real data.")

        ov_inventory = st.number_input(
            "Inventory Level Override", min_value=0, value=0, step=10,
            help="Override inventory units fed into the model (0 = use actual)"
        )
        ov_price = st.number_input(
            "Price Override ($)", min_value=0.0, value=0.0, step=1.0,
            help="Override unit price (0 = use actual)"
        )
        ov_discount = st.number_input(
            "Discount Override (%)", min_value=0, max_value=50, value=0, step=1,
            help="Override discount percentage (0 = use actual)"
        )
        ov_promo = st.selectbox(
            "Promotion Override",
            ["Use Actual", "Promotion ON (1)", "Promotion OFF (0)"],
        )

        run_clicked = st.button("🚀 Run Forecast", type="primary", use_container_width=True)

        st.divider()
        st.subheader("📊 Model Metrics (test set)")
        st.metric("WAPE", f"{metrics['WAPE']:.2f}%", help="Weighted Absolute Percentage Error")
        c1, c2 = st.columns(2)
        c1.metric("MAE",  f"{metrics['MAE']:.0f}")
        c2.metric("RMSE", f"{metrics['RMSE']:.0f}")
        st.caption("Trained weekly at store-category level — best granularity for this dataset.")

    # ─────────────────────────────────────────
    # Build overrides dict
    # ─────────────────────────────────────────
    overrides: dict = {}
    if ov_inventory > 0:
        overrides["Inventory_Level"] = float(ov_inventory)
    if ov_price > 0.0:
        overrides["Price"] = float(ov_price)
    if ov_discount > 0:
        overrides["Discount"] = float(ov_discount)
    if "ON" in ov_promo:
        overrides["Holiday/Promotion"] = 1
    elif "OFF" in ov_promo:
        overrides["Holiday/Promotion"] = 0

    # ─────────────────────────────────────────
    # Run forecast (auto on first load, or when button pressed)
    # ─────────────────────────────────────────
    state_key = f"{store_filter}|{category_filter}|{n_weeks}|{overrides}"
    need_rerun = (
        run_clicked
        or "forecast_df" not in st.session_state
        or st.session_state.get("state_key") != state_key
    )

    if need_rerun:
        with st.spinner("Generating forecasts…"):
            result_df = forecast_weeks(
                df, model_data,
                n_weeks=n_weeks,
                store_filter=store_filter if store_filter != "All" else None,
                category_filter=category_filter if category_filter != "All" else None,
                overrides=overrides or None,
            )
        st.session_state["forecast_df"] = result_df
        st.session_state["state_key"]   = state_key
        st.session_state["llm_summary"] = None   # reset summary on new forecast

    forecast_df: pd.DataFrame = st.session_state["forecast_df"]

    if forecast_df is None or forecast_df.empty:
        st.warning("No data returned for this combination. Try different filters.")
        return

    latest = forecast_df[forecast_df["Week"] == forecast_df["Week"].max()].copy()

    # ─────────────────────────────────────────
    # KPI cards
    # ─────────────────────────────────────────
    grow_n    = int((latest["Decision_Tier"] == "Grow").sum())
    monitor_n = int((latest["Decision_Tier"] == "Monitor").sum())
    reduce_n  = int((latest["Decision_Tier"] == "Reduce").sum())
    avg_dem   = latest["Forecasted_Demand"].mean()

    c1, c2, c3, c4, c5 = st.columns(5)
    c1.metric("Products Analysed", len(latest))
    c2.metric("🟢 Grow",    grow_n,    f"+{grow_n} need restock")
    c3.metric("🟡 Monitor", monitor_n)
    c4.metric("🔴 Reduce",  reduce_n,  f"−{reduce_n} overstock", delta_color="inverse")
    c5.metric("Avg Forecasted Demand", f"{avg_dem:.0f} u/wk")

    st.divider()

    # ─────────────────────────────────────────
    # Tabs
    # ─────────────────────────────────────────
    tab1, tab2, tab3, tab4 = st.tabs(["📋 Forecast Table", "📊 Charts & Trends", "🔬 Model Explainability", "🤖 AI Insights"])

    with tab1:
        _tab_table(forecast_df, n_weeks)

    with tab2:
        _tab_charts(forecast_df, latest, n_weeks)

    with tab3:
        _tab_explainability(model_data)

    with tab4:
        _tab_ai(forecast_df, latest, store_filter, n_weeks)


# ─────────────────────────────────────────────
# Tab 1 — Table
# ─────────────────────────────────────────────
def _tab_table(forecast_df: pd.DataFrame, n_weeks: int):
    st.subheader("Demand Forecast Results")

    if n_weeks > 1:
        week_sel = st.radio(
            "Select week to display",
            list(range(1, n_weeks + 1)),
            format_func=lambda x: f"Week +{x}",
            horizontal=True,
        )
        display = forecast_df[forecast_df["Week"] == week_sel].copy()
    else:
        display = forecast_df.copy()

    cols = [
        "Store ID", "Product ID", "Category",
        "Inventory Level", "Remaining_Inventory",
        "Forecasted_Demand", "Coverage_Ratio",
        "DSG", "STS", "PIS", "Decision_Score",
        "Decision_Tier", "Primary_Driver",
    ]
    display = display[cols].rename(columns={
        "Remaining_Inventory": "Remaining Inv.",
        "Forecasted_Demand":   "Forecast (units)",
        "Coverage_Ratio":      "Cov. Ratio",
        "Decision_Score":      "Score",
        "Decision_Tier":       "Tier",
        "Primary_Driver":      "Key Driver",
    })

    # Round numerics
    for col in ["Forecast (units)", "Cov. Ratio", "DSG", "STS", "PIS", "Score"]:
        display[col] = display[col].round(1)
    display["Remaining Inv."] = display["Remaining Inv."].round(0).astype(int)

    # Apply tier colour highlighting
    def highlight_tier(val):
        bg = TIER_BG.get(val, "#f9fafb")
        fg = TIER_FG.get(val, "#111827")
        return f"background-color: {bg}; color: {fg}; font-weight: 600"

    styled = display.style.applymap(highlight_tier, subset=["Tier"])
    st.dataframe(styled, use_container_width=True, height=440)

    csv = display.to_csv(index=False)
    st.download_button(
        "📥 Download CSV", csv,
        file_name="demand_forecast.csv",
        mime="text/csv",
    )


# ─────────────────────────────────────────────
# Tab 2 — Charts
# ─────────────────────────────────────────────
def _tab_charts(forecast_df: pd.DataFrame, latest: pd.DataFrame, n_weeks: int):
    # Row 1
    col1, col2 = st.columns([3, 2])

    with col1:
        st.subheader("Forecasted Demand by Product")
        top20 = latest.nlargest(20, "Forecasted_Demand")
        fig = px.bar(
            top20, x="Product ID", y="Forecasted_Demand",
            color="Decision_Tier",
            color_discrete_map=TIER_COLORS,
            hover_data=["Store ID", "Category", "Coverage_Ratio", "Decision_Score"],
            labels={"Forecasted_Demand": "Units (next week)", "Product ID": "Product"},
            title="Top 20 Products — Forecasted Demand",
        )
        fig.update_layout(height=400, legend_title_text="Tier")
        st.plotly_chart(fig, use_container_width=True)

    with col2:
        st.subheader("Decision Tier Distribution")
        tier_counts = latest["Decision_Tier"].value_counts().reset_index()
        tier_counts.columns = ["Tier", "Count"]
        fig2 = px.pie(
            tier_counts, values="Count", names="Tier",
            color="Tier", color_discrete_map=TIER_COLORS,
            title="Tier Breakdown (latest forecast week)",
            hole=0.4,
        )
        fig2.update_traces(textinfo="percent+label")
        fig2.update_layout(height=400, showlegend=False)
        st.plotly_chart(fig2, use_container_width=True)

    # Row 2 — multi-week trend (only if n_weeks > 1)
    if n_weeks > 1:
        st.subheader("Demand Trend Across Forecast Weeks")
        trend = (
            forecast_df.groupby(["Week", "Category"])["Forecasted_Demand"]
            .sum().reset_index()
        )
        fig3 = px.line(
            trend, x="Week", y="Forecasted_Demand",
            color="Category", markers=True,
            labels={"Forecasted_Demand": "Total Forecasted Units", "Week": "Week Ahead"},
            title="Total Forecasted Demand per Category (Weeks +1 to +3)",
        )
        fig3.update_xaxes(tickvals=list(range(1, n_weeks + 1)),
                           ticktext=[f"Week +{w}" for w in range(1, n_weeks + 1)])
        fig3.update_layout(height=360)
        st.plotly_chart(fig3, use_container_width=True)

    # Row 3
    col3, col4 = st.columns(2)

    with col3:
        st.subheader("Avg Decision Score by Category")
        cat_scores = (
            latest.groupby("Category")["Decision_Score"].mean().reset_index()
              .sort_values("Decision_Score", ascending=False)
        )
        fig4 = px.bar(
            cat_scores, x="Category", y="Decision_Score",
            color="Decision_Score",
            color_continuous_scale=["#ef4444", "#f59e0b", "#22c55e"],
            range_color=[0, 100],
            labels={"Decision_Score": "Avg Score (0–100)"},
            title="Average Decision Intelligence Score by Category",
        )
        fig4.add_hline(y=70, line_dash="dash", line_color="#22c55e",
                       annotation_text="Grow ≥70", annotation_position="top right")
        fig4.add_hline(y=40, line_dash="dash", line_color="#ef4444",
                       annotation_text="Reduce <40", annotation_position="bottom right")
        fig4.update_layout(height=380, coloraxis_showscale=False)
        st.plotly_chart(fig4, use_container_width=True)

    with col4:
        st.subheader("Inventory vs Forecasted Demand")
        fig5 = px.scatter(
            latest, x="Inventory Level", y="Forecasted_Demand",
            color="Decision_Tier", color_discrete_map=TIER_COLORS,
            hover_data=["Store ID", "Product ID", "Category", "Coverage_Ratio"],
            labels={"Forecasted_Demand": "Forecast (units)", "Inventory Level": "Inventory"},
            title="Inventory vs Demand (dot = one product)",
        )
        max_v = max(float(latest["Inventory Level"].max()),
                    float(latest["Forecasted_Demand"].max())) * 1.05
        fig5.add_trace(go.Scatter(
            x=[0, max_v], y=[0, max_v], mode="lines",
            name="Coverage = 1×", line=dict(dash="dash", color="gray", width=1),
        ))
        fig5.update_layout(height=380, legend_title_text="Tier")
        st.plotly_chart(fig5, use_container_width=True)

    # Row 4 — Score component breakdown (stacked bar)
    st.subheader("Decision Score Component Breakdown (per Store)")
    comp = (
        latest.groupby("Store ID")[["DSG", "STS", "PIS"]].mean().reset_index()
    )
    comp_melt = comp.melt(id_vars="Store ID", var_name="Component", value_name="Score")
    component_colors = {"DSG": "#6366f1", "STS": "#f59e0b", "PIS": "#10b981"}
    fig6 = px.bar(
        comp_melt, x="Store ID", y="Score", color="Component",
        color_discrete_map=component_colors, barmode="group",
        labels={"Score": "Avg Component Score (0–100)"},
        title="Avg DSG / STS / PIS Score by Store  (DSG=Demand-Supply Gap · STS=Sales Trend · PIS=Promo Impact)",
    )
    fig6.update_layout(height=360)
    st.plotly_chart(fig6, use_container_width=True)


# ─────────────────────────────────────────────
# Tab 3 — Model Explainability
# ─────────────────────────────────────────────
def _tab_explainability(md: dict):
    st.subheader("🔬 Feature Importance (LightGBM Gain)")
    st.caption(
        "Shows how much each input feature contributes to the model's predictions "
        "(higher = more influential). Mirrors a SHAP bar chart."
    )

    importance_df = md["importance_df"]
    fig = px.bar(
        importance_df,
        x="Importance",
        y="Feature Label",
        orientation="h",
        color="Importance",
        color_continuous_scale=["#c7d2fe", "#4f46e5"],
        labels={"Importance": "Feature Importance (Gain)", "Feature Label": ""},
        title="Feature Importance — What Drives the Forecast?",
    )
    fig.update_layout(
        height=420,
        coloraxis_showscale=False,
        yaxis={"categoryorder": "total ascending"},
    )
    st.plotly_chart(fig, use_container_width=True)

    st.divider()

    # ── Actual vs Predicted ──
    st.subheader("📈 Forecast vs Actual (Test Set)")
    st.caption(
        "Each dot is one store-category-week in the held-out test period. "
        "Points on the diagonal line = perfect prediction."
    )

    test_plot = md["test_plot"].copy()

    col1, col2 = st.columns([2, 1])

    with col1:
        # Scatter: Actual vs Predicted
        fig2 = px.scatter(
            test_plot,
            x="Actual",
            y="Predicted",
            color="Category",
            hover_data=["Store ID", "Week"],
            opacity=0.65,
            labels={"Actual": "Actual Units Sold", "Predicted": "Predicted Units"},
            title="Actual vs Predicted — Test Set",
        )
        max_val = float(max(test_plot["Actual"].max(), test_plot["Predicted"].max())) * 1.05
        fig2.add_trace(go.Scatter(
            x=[0, max_val], y=[0, max_val],
            mode="lines", name="Perfect prediction",
            line=dict(dash="dash", color="gray", width=1),
        ))
        fig2.update_layout(height=430)
        st.plotly_chart(fig2, use_container_width=True)

    with col2:
        # Error distribution
        test_plot["Error %"] = ((test_plot["Predicted"] - test_plot["Actual"]) /
                                 test_plot["Actual"].replace(0, np.nan) * 100).clip(-100, 100)
        fig3 = px.histogram(
            test_plot, x="Error %", nbins=40,
            color_discrete_sequence=["#6366f1"],
            title="Prediction Error Distribution",
            labels={"Error %": "Error % (Predicted − Actual) / Actual"},
        )
        fig3.add_vline(x=0, line_dash="dash", line_color="gray")
        fig3.update_layout(height=430)
        st.plotly_chart(fig3, use_container_width=True)

    # Time-series view: aggregate actual vs predicted per week
    st.subheader("📅 Weekly Aggregate — Actual vs Predicted (Test Period)")
    weekly_agg = test_plot.groupby("Week")[["Actual", "Predicted"]].sum().reset_index()
    fig4 = go.Figure()
    fig4.add_trace(go.Scatter(
        x=weekly_agg["Week"], y=weekly_agg["Actual"],
        mode="lines+markers", name="Actual",
        line=dict(color="#22c55e", width=2),
    ))
    fig4.add_trace(go.Scatter(
        x=weekly_agg["Week"], y=weekly_agg["Predicted"],
        mode="lines+markers", name="Predicted",
        line=dict(color="#6366f1", width=2, dash="dash"),
    ))
    fig4.update_layout(
        height=380,
        xaxis_title="Week",
        yaxis_title="Total Units Sold (all store-categories)",
        legend_title="",
        hovermode="x unified",
        title="Total Weekly Sales — Actual vs Predicted",
    )
    st.plotly_chart(fig4, use_container_width=True)

    # Metrics summary
    st.divider()
    st.subheader("📊 Test-Set Model Metrics")
    m = md["metrics"]
    c1, c2, c3 = st.columns(3)
    c1.metric("WAPE", f"{m['WAPE']:.2f}%", help="Weighted Absolute Percentage Error — lower is better")
    c2.metric("MAE", f"{m['MAE']:.0f} units", help="Mean Absolute Error per store-category-week")
    c3.metric("RMSE", f"{m['RMSE']:.0f} units", help="Root Mean Squared Error — penalises large misses more")
    st.caption(
        f"Benchmark: naive historical-average baseline achieves ~21.8% WAPE on this dataset. "
        f"LightGBM at **{m['WAPE']:.2f}%** represents a meaningful improvement."
    )


# ─────────────────────────────────────────────
# Tab 4 — AI Insights
# ─────────────────────────────────────────────
def _tab_ai(forecast_df: pd.DataFrame, latest: pd.DataFrame,
             store_filter: str, n_weeks: int):
    st.subheader("🤖 AI-Generated Executive Summary")
    st.caption("Powered by Google  Gemini AI Summaries · summarises the current forecast")

    # LLM summary section
    if st.session_state.get("llm_summary") is None:
        col_btn, _ = st.columns([1, 3])
        with col_btn:
            if st.button("✨ Generate AI Summary", type="primary", use_container_width=True):
                with st.spinner("Gemini is reading your forecast data…"):
                    summary = generate_summary(forecast_df, store_filter, n_weeks)
                st.session_state["llm_summary"] = summary
                st.rerun()
    else:
        st.info(st.session_state["llm_summary"])
        if st.button("🔄 Regenerate"):
            st.session_state["llm_summary"] = None
            st.rerun()

    st.divider()

    # ── Q&A Chat ──
    st.subheader("💬 Ask a Question About Your Forecast")
    st.caption("Ask anything about the current forecast — tiers, products, categories, inventory risk, recommendations.")

    if "chat_history" not in st.session_state:
        st.session_state["chat_history"] = []

    # Render existing chat history
    for msg in st.session_state["chat_history"]:
        with st.chat_message(msg["role"]):
            st.write(msg["content"])

    question = st.chat_input("e.g. Which products need urgent restocking? / What is the demand for Electronics?")

    if question:
        # Show user message immediately
        st.session_state["chat_history"].append({"role": "user", "content": question})
        with st.chat_message("user"):
            st.write(question)

        # Get and show AI answer
        with st.chat_message("assistant"):
            with st.spinner("Thinking…"):
                answer = answer_question(question, forecast_df, store_filter, n_weeks)
            st.write(answer)
        st.session_state["chat_history"].append({"role": "assistant", "content": answer})

    if st.session_state["chat_history"]:
        if st.button("🗑️ Clear chat", key="clear_chat"):
            st.session_state["chat_history"] = []
            st.rerun()

    st.divider()

    # ── Alert cards ──
    st.subheader("🚨 Product Alerts")
    col_g, col_r = st.columns(2)

    with col_g:
        st.markdown("#### 🟢 Top GROW — Restock Urgently")
        grow = (
            latest[latest["Decision_Tier"] == "Grow"]
            .nlargest(6, "Decision_Score")
        )
        if grow.empty:
            st.info("No Grow-tier products in current filter.")
        else:
            for _, r in grow.iterrows():
                st.success(
                    f"**{r['Store ID']} / {r['Product ID']}** — {r['Category']}  \n"
                    f"Forecast: **{r['Forecasted_Demand']:.0f} units** · "
                    f"Coverage: {r['Coverage_Ratio']:.2f}× · "
                    f"Score: {r['Decision_Score']:.0f}"
                )

    with col_r:
        st.markdown("#### 🔴 Top REDUCE — Overstock Risk")
        red = (
            latest[latest["Decision_Tier"] == "Reduce"]
            .nsmallest(6, "Decision_Score")
        )
        if red.empty:
            st.info("No Reduce-tier products in current filter.")
        else:
            for _, r in red.iterrows():
                st.error(
                    f"**{r['Store ID']} / {r['Product ID']}** — {r['Category']}  \n"
                    f"Forecast: **{r['Forecasted_Demand']:.0f} units** · "
                    f"Coverage: {r['Coverage_Ratio']:.2f}× · "
                    f"Score: {r['Decision_Score']:.0f}"
                )

    st.divider()

    # ── Primary driver summary ──
    st.subheader("📌 What's Driving the Scores?")
    driver_counts = latest["Primary_Driver"].value_counts().reset_index()
    driver_counts.columns = ["Driver", "Products Affected"]
    fig = px.bar(
        driver_counts, x="Products Affected", y="Driver",
        orientation="h", color="Products Affected",
        color_continuous_scale="Blues",
        title="Most Common Decision Intelligence Drivers",
    )
    fig.update_layout(height=350, coloraxis_showscale=False, yaxis={"categoryorder": "total ascending"})
    st.plotly_chart(fig, use_container_width=True)


if __name__ == "__main__":
    main()


Writing app.py


In [5]:
%%writefile llm_summary.py
"""
LLM summarization using Google Gemini API.
"""
import os
import pandas as pd


def generate_summary(forecast_df: pd.DataFrame, store_id: str, n_weeks: int) -> str:
    """
    Generate a narrative executive summary of the forecast using Gemini.
    Falls back to a rule-based summary if the API key is missing or call fails.
    """
    api_key = os.environ.get("GEMINI_API_KEY", "").strip()

    # Prepare data for the prompt
    latest = forecast_df[forecast_df["Week"] == forecast_df["Week"].max()].copy()
    tier_dist = latest["Decision_Tier"].value_counts().to_dict()
    grow_count = tier_dist.get("Grow", 0)
    monitor_count = tier_dist.get("Monitor", 0)
    reduce_count = tier_dist.get("Reduce", 0)
    total = len(latest)

    top_grow = (
        latest[latest["Decision_Tier"] == "Grow"]
        .nlargest(3, "Decision_Score")[["Store ID", "Product ID", "Category",
                                         "Forecasted_Demand", "Decision_Score"]]
    )
    top_reduce = (
        latest[latest["Decision_Tier"] == "Reduce"]
        .nsmallest(3, "Decision_Score")[["Store ID", "Product ID", "Category",
                                          "Forecasted_Demand", "Coverage_Ratio"]]
    )

    avg_demand = latest["Forecasted_Demand"].mean()
    cat_demand = latest.groupby("Category")["Forecasted_Demand"].mean().round(1).to_dict()

    grow_text = top_grow.to_string(index=False) if not top_grow.empty else "None"
    reduce_text = top_reduce.to_string(index=False) if not top_reduce.empty else "None"

    weeks_label = f"{n_weeks} week(s)"
    store_label = store_id if store_id != "All" else "all stores"

    prompt = f"""You are a senior retail analytics consultant. Provide a concise, data-driven executive summary of the following sales demand forecast.

Forecast Details:
- Scope: {store_label}
- Horizon: next {weeks_label}
- Total products analysed: {total}

Decision Tier Breakdown:
- Grow (invest / restock urgently): {grow_count} products
- Monitor (watch closely): {monitor_count} products
- Reduce (overstock risk — slow movement): {reduce_count} products

Average forecasted weekly demand per product: {avg_demand:.1f} units

Average demand by category:
{cat_demand}

Top products to GROW (understock risk):
{grow_text}

Top products to REDUCE (overstock risk):
{reduce_text}

Write a professional executive summary of exactly 4 sentences:
1. Overall demand outlook and key trend.
2. Which categories or products need urgent restocking.
3. Which categories or products face overstock risk.
4. One clear, actionable inventory management recommendation.

Do NOT use bullet points. Write in flowing prose. Be specific with numbers."""

    if api_key:
        try:
            import google.generativeai as genai  # type: ignore
            genai.configure(api_key=api_key)
            model = genai.GenerativeModel("gemini-flash-latest")
            response = model.generate_content(prompt)
            return response.text.strip()
        except Exception as exc:
            return _rule_based_summary(
                store_label, weeks_label, total, grow_count, monitor_count,
                reduce_count, avg_demand, top_grow, top_reduce, error=str(exc)
            )

    return _rule_based_summary(
        store_label, weeks_label, total, grow_count, monitor_count,
        reduce_count, avg_demand, top_grow, top_reduce
    )


def answer_question(question: str, forecast_df: pd.DataFrame, store_id: str, n_weeks: int) -> str:
    """
    Answer a user question about the current forecast using Gemini.
    Falls back gracefully when Gemini is unavailable or the question
    cannot be answered from the available data.
    """
    api_key = os.environ.get("GEMINI_API_KEY", "").strip()

    if not question.strip():
        return "Please type a question first."

    # Build a rich data context for the model
    latest = forecast_df[forecast_df["Week"] == forecast_df["Week"].max()].copy()
    tier_dist = latest["Decision_Tier"].value_counts().to_dict()
    cat_demand = latest.groupby("Category")["Forecasted_Demand"].mean().round(1).to_dict()
    cat_tier = (
        latest.groupby(["Category", "Decision_Tier"])
        .size().unstack(fill_value=0).to_string()
    )
    top_grow = (
        latest[latest["Decision_Tier"] == "Grow"]
        .nlargest(5, "Decision_Score")[
            ["Store ID", "Product ID", "Category", "Forecasted_Demand",
             "Coverage_Ratio", "Decision_Score", "Primary_Driver"]
        ].to_string(index=False)
    )
    top_reduce = (
        latest[latest["Decision_Tier"] == "Reduce"]
        .nsmallest(5, "Decision_Score")[
            ["Store ID", "Product ID", "Category", "Forecasted_Demand",
             "Coverage_Ratio", "Decision_Score", "Primary_Driver"]
        ].to_string(index=False)
    )
    top_monitor = (
        latest[latest["Decision_Tier"] == "Monitor"]
        .nlargest(5, "Decision_Score")[
            ["Store ID", "Product ID", "Category", "Forecasted_Demand",
             "Coverage_Ratio", "Decision_Score"]
        ].to_string(index=False)
    )
    avg_score = latest["Decision_Score"].mean()
    avg_demand = latest["Forecasted_Demand"].mean()
    store_label = store_id if store_id != "All" else "all stores"

    context = f"""You are an AI assistant embedded in a retail sales demand forecasting dashboard.
Answer the user's question using ONLY the forecast data provided below.

=== FORECAST CONTEXT ===
Scope: {store_label} | Horizon: {n_weeks} week(s) | Total products: {len(latest)}
Tier breakdown: {tier_dist}
Average forecasted demand per product: {avg_demand:.1f} units/week
Average decision score: {avg_score:.1f} / 100

Average demand by category:
{cat_demand}

Tier counts by category:
{cat_tier}

Top GROW products (restock urgently):
{top_grow}

Top REDUCE products (overstock risk):
{top_reduce}

Top MONITOR products:
{top_monitor}

=== RULES ===
- Answer only from the data above. Do not make up figures not present in the data.
- If the question cannot be answered from this data, reply with exactly:
  "I can't answer your question with the given forecast data. Try asking about product tiers, demand forecasts, coverage ratios, or inventory recommendations."
- Be concise and specific. Use numbers from the data. No bullet points unless listing items.
- If the question is a greeting or off-topic, politely redirect to the forecast data.

=== USER QUESTION ===
{question}"""

    if api_key:
        try:
            import google.generativeai as genai  # type: ignore
            genai.configure(api_key=api_key)
            model = genai.GenerativeModel("gemini-flash-latest")
            response = model.generate_content(context)
            return response.text.strip()
        except Exception as exc:
            return (
                f"I can't answer your question right now — Gemini returned an error: {str(exc)[:120]}. "
                "Please try again in a moment."
            )

    return (
        "I can't answer your question with the given forecast data — "
        "the Gemini API key is not configured. Please add your GEMINI_API_KEY secret and try again."
    )


def _rule_based_summary(
    store_label, weeks_label, total, grow_count, monitor_count,
    reduce_count, avg_demand, top_grow, top_reduce, error: str = ""
) -> str:
    grow_pct = round(grow_count / total * 100) if total else 0
    reduce_pct = round(reduce_count / total * 100) if total else 0

    grow_str = ""
    if not top_grow.empty:
        r = top_grow.iloc[0]
        grow_str = (
            f" Highest urgency: {r['Store ID']}/{r['Product ID']} ({r['Category']}) "
            f"with {r['Forecasted_Demand']:.0f} units forecasted and a Decision Score of {r['Decision_Score']:.0f}."
        )

    reduce_str = ""
    if not top_reduce.empty:
        r = top_reduce.iloc[0]
        reduce_str = (
            f" Highest overstock risk: {r['Store ID']}/{r['Product ID']} ({r['Category']}) "
            f"with a coverage ratio of {r['Coverage_Ratio']:.2f}x."
        )

    note = f" (Gemini unavailable — {error[:60]})" if error else ""

    return (
        f"The demand forecast for {store_label} over the next {weeks_label} shows an average of "
        f"{avg_demand:.0f} units per product, with {grow_pct}% of SKUs in the Grow tier and "
        f"{reduce_pct}% flagged for reduction.{grow_str}"
        f" {reduce_count} product(s) carry overstock risk and should have orders paused or promotions applied.{reduce_str}"
        f" Recommendation: prioritise restocking Grow-tier products within 48 hours and consider "
        f"markdown discounts or inter-store transfers for Reduce-tier items.{note}"
    )


Writing llm_summary.py


In [6]:
# ── Standalone: get forecast CSV directly in Colab ──
from model import load_data, train_model, forecast_weeks
from google.colab import files

print("Loading data & training model…")
df = load_data()
md = train_model(df)
print(f"Model trained — WAPE: {md['metrics']['WAPE']:.2f}%")

# Change n_weeks to 1, 2, or 3
forecast_df = forecast_weeks(df, md, n_weeks=3)

# Keep the most useful columns
output = forecast_df[[
    "Store ID", "Product ID", "Category", "Week",
    "Inventory Level", "Remaining_Inventory",
    "Forecasted_Demand", "Coverage_Ratio",
    "DSG", "STS", "PIS", "Decision_Score",
    "Decision_Tier", "Primary_Driver"
]].rename(columns={
    "Remaining_Inventory":  "Remaining_Inventory_After_Depletion",
    "Forecasted_Demand":    "Forecasted_Demand_Units",
    "Coverage_Ratio":       "Coverage_Ratio",
    "Decision_Tier":        "Tier_Grow_Monitor_Reduce",
})

output = output.sort_values(["Store ID", "Product ID", "Week"])
output.to_csv("demand_forecast.csv", index=False)
print(f"Saved {len(output)} rows  ({output['Week'].nunique()} weeks × {len(output)//output['Week'].nunique()} products)")
print(output["Tier_Grow_Monitor_Reduce"].value_counts().to_string())

Loading data & training model…
Model trained — WAPE: 18.79%
Saved 300 rows  (3 weeks × 100 products)
Tier_Grow_Monitor_Reduce
Monitor    197
Grow        79
Reduce      24


In [7]:
!pip install psycopg2-binary sqlalchemy
import os
import getpass
from sqlalchemy import create_engine

host = getpass.getpass("Enter Supabase host: ")
port = getpass.getpass("Enter Supabase port [6543]: ") or "6543"
user = getpass.getpass("Enter Supabase username: ")
password = getpass.getpass("Enter Supabase password: ")
database = getpass.getpass("Enter database name [postgres]: ") or "postgres"

engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}")

output.to_sql(
    name="demand_forecast",
    con=engine,
    if_exists="replace",
    index=False
)
print("Uploaded to Supabase ✅")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 26.5 MB/s eta 0:00:00
Enter Supabase host: ··········
Enter Supabase port [6543]: ··········
Enter Supabase username: ··········
Enter Supabase password: ··········
Enter database name [postgres]: ··········
Uploaded to Supabase ✅


In [9]:
import os
import getpass
from pyngrok import ngrok

# Prompt for Gemini API key (hidden while typing)
GEMINI_API_KEY = getpass.getpass("Enter your Gemini API key: ")
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

# Prompt for ngrok auth token (hidden while typing)
NGROK_TOKEN = getpass.getpass("Enter your ngrok auth token: ")
ngrok.set_auth_token(NGROK_TOKEN)

# Start ngrok tunnel
public_url = ngrok.connect(8501)
print("🚀 App URL:", public_url)

# Run Streamlit
!streamlit run app.py --server.port 8501 --server.headless true

Enter your Gemini API key: ··········
Enter your ngrok auth token: ··········
🚀 App URL: NgrokTunnel: "https://conclude-reassure-venomous.ngrok-free.dev" -> "http://localhost:8501"


2026-08-08 15:28:29.167 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.67.121.155:8501

/content/app.py:245: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styled = display.style.applymap(highlight_tier, subset=["Tier"])
2026-08-08 15:28:39.665 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-08-08 15:28:40.484 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width=